In [ ]:
# %%
import warnings
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

BASE_PATH = '/Users/egorilin/Desktop/COMA'
BASE = Path(BASE_PATH)

RANDOM_STATE = 42
TEST_SIZE = 0.2

CB_KW = dict(iterations=500, learning_rate=0.05, depth=6,
             verbose=False, random_seed=RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

TARGET_COLUMNS = [
    'S. aureus ATCC 43300 Activity MIC, мг/л',
    'S. aureus ATCC 43300 Activity MBC, мг/л',
    'E. coli ATCC 25922 Activity MIC, мг/л',
    'E. coli ATCC 25922 Activity MBC, мг/л',
]

# 'biocides' — 12 фич (без диполей); 'conc' — 15 фич (с диполями)
DATASET_MODE = 'conc'

FEATURE_COLS_FULL = [
    'Molecular weight', 'LogP', 'mmff94', 'gasteiger', 'eem2015bm',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]
FEATURE_COLS_BIO = [
    'Molecular weight', 'LogP',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]
FEATURE_COLS = FEATURE_COLS_BIO if DATASET_MODE == 'biocides' else FEATURE_COLS_FULL
DIPOLE_COLS = ['mmff94', 'gasteiger', 'eem2015bm']

DESC_OUT_DIR = BASE / 'descriptors_test_predictions'
EMB_OUT_DIR  = BASE / 'embeddings_test_predictions'
DESC_OUT_DIR.mkdir(exist_ok=True)
EMB_OUT_DIR.mkdir(exist_ok=True)

print(f'Dataset mode: {DATASET_MODE}  |  {len(FEATURE_COLS)} фич')
print(f'Output dirs:\n  {DESC_OUT_DIR}\n  {EMB_OUT_DIR}')

In [ ]:
target

In [ ]:
# %%
target = pd.read_excel(f'{BASE_PATH}/Data_biocides.xlsx')
desc_df = pd.read_excel(f'{BASE_PATH}/Data_prepare_biocides_molformer_new_embeddings.xlsx')
data_drop = pd.read_excel(f'{BASE_PATH}/Test_molformer_original_biocides.xlsx').drop(
    ['Unnamed: 0'], axis=1
)
dipoles_df = pd.read_csv(f'{BASE_PATH}/Train_dipole_moments.csv')

print(f'target:     {target.shape}')
print(f'desc_df:    {desc_df.shape}')
print(f'data_drop:  {data_drop.shape}')
print(f'dipoles_df: {dipoles_df.shape}  | cols: {list(dipoles_df.columns)}')

# --- Базовые фичи (LogP, Molecular weight) берём из target по позиции ---
BASE_FEATURE_COLS = ['Molecular weight', 'LogP']
missing_in_target = [c for c in BASE_FEATURE_COLS if c not in target.columns]
if missing_in_target:
    raise ValueError(f'В target нет колонок: {missing_in_target}. '
                     f'Колонки target: {list(target.columns)[:20]}')

desc_df = desc_df.reset_index(drop=True)
target_reset = target.reset_index(drop=True)

# выкидываем эти колонки из desc_df, если уже есть, и подставляем из target по позиции
desc_df = desc_df.drop(columns=[c for c in BASE_FEATURE_COLS if c in desc_df.columns])
n_base = min(len(desc_df), len(target_reset))
if len(desc_df) != len(target_reset):
    print(f'  ⚠️  Длины desc_df({len(desc_df)}) и target({len(target_reset)}) не совпадают, '
          f'возьму первые {n_base}')

for c in BASE_FEATURE_COLS:
    desc_df[c] = np.nan
    desc_df.loc[:n_base - 1, c] = target_reset[c].iloc[:n_base].values

print('\nПодставлено из target:')
for c in BASE_FEATURE_COLS:
    n_nan = desc_df[c].isna().sum()
    print(f'  {c}: {len(desc_df) - n_nan}/{len(desc_df)} заполнено  ({n_nan} NaN)')

# --- Подстановка диполей по позиции из csv ---
first_col = dipoles_df.columns[0]
if dipoles_df[first_col].dtype.kind in 'iu' or first_col.lower() in ('unnamed: 0', 'index', 'id', '№'):
    print(f'\n  Отбрасываю служебную первую колонку csv: "{first_col}"')
    dipoles_df = dipoles_df.drop(columns=[first_col])

have_dipoles = [c for c in DIPOLE_COLS if c in dipoles_df.columns]
miss_dipoles = [c for c in DIPOLE_COLS if c not in dipoles_df.columns]
print(f'  Диполи в csv: {have_dipoles}')
if miss_dipoles:
    print(f'  ⚠️  Нет в csv диполей: {miss_dipoles}')

n_dip = min(len(desc_df), len(dipoles_df))
if len(dipoles_df) != len(desc_df):
    print(f'  ⚠️  Длины desc_df({len(desc_df)}) и dipoles_df({len(dipoles_df)}) не совпадают, '
          f'возьму первые {n_dip}')

desc_df = desc_df.drop(columns=[c for c in DIPOLE_COLS if c in desc_df.columns])
dipoles_aligned = dipoles_df[have_dipoles].reset_index(drop=True)

for c in have_dipoles:
    desc_df[c] = np.nan
    desc_df.loc[:n_dip - 1, c] = dipoles_aligned[c].iloc[:n_dip].values

print('\nПокрытие диполей после подстановки:')
for c in have_dipoles:
    n_nan = desc_df[c].isna().sum()
    print(f'  {c}: {len(desc_df) - n_nan}/{len(desc_df)} заполнено  ({n_nan} NaN)')

# --- Таргеты тоже должны быть в desc_df для Ячейки 4. Если их там нет — берём из target по позиции ---
for c in TARGET_COLUMNS:
    if c not in desc_df.columns:
        if c not in target_reset.columns:
            raise ValueError(f'Нет таргета "{c}" ни в desc_df, ни в target')
        desc_df[c] = np.nan
        n_tg = min(len(desc_df), len(target_reset))
        desc_df.loc[:n_tg - 1, c] = target_reset[c].iloc[:n_tg].values

# --- Финальные проверки ---
missing_desc = [c for c in FEATURE_COLS if c not in desc_df.columns]
if missing_desc:
    raise ValueError(f'Нет колонок в desc_df после всех подстановок: {missing_desc}')

for c in TARGET_COLUMNS:
    if c not in target.columns:
        raise ValueError(f'Нет таргета в target: {c}')

print('\n✓ Все нужные колонки на месте')
print(f'  FEATURE_COLS ({len(FEATURE_COLS)}): {FEATURE_COLS}')

In [ ]:
# %%
class AdvancedContrastiveEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 384, 128],
                 projection_dim=128, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hidden_dim
        self.encoder = nn.Sequential(*layers)
        self.projection_head = nn.Sequential(
            nn.Linear(hidden_dims[-1], projection_dim),
            nn.BatchNorm1d(projection_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(projection_dim, projection_dim),
            nn.BatchNorm1d(projection_dim),
        )

    def forward(self, x, return_projection=True):
        features = self.encoder(x)
        if return_projection:
            return features, self.projection_head(features)
        return features


encoder = AdvancedContrastiveEncoder(
    input_dim=768, hidden_dims=[512, 384, 256],
    projection_dim=128, dropout=0.3,
).to(device)
encoder.load_state_dict(
    torch.load(f'{BASE_PATH}/encoder_borderline_smote.pth', map_location=device)
)
encoder.eval()

with open(f'{BASE_PATH}/scaler_classification.pkl', 'rb') as f:
    scaler = pickle.load(f)

data_scaled = scaler.transform(data_drop)
X_tensor = torch.FloatTensor(data_scaled).to(device)
with torch.no_grad():
    molecular_embeddings = encoder(X_tensor, return_projection=False).cpu().numpy()

print(f'✓ molecular_embeddings: {molecular_embeddings.shape}')

In [ ]:
# %%
desc_all_records  = []
desc_metrics_rows = []

for target_col in TARGET_COLUMNS:
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    tag = f"{organism.replace(' ', '_').replace('.', '')}_{short_name}"

    cols_needed = FEATURE_COLS + [target_col]
    valid_mask = desc_df[cols_needed].notna().all(axis=1)
    sub = desc_df.loc[valid_mask, cols_needed].reset_index().rename(
        columns={'index': 'mol_idx'}
    )

    if len(sub) < 10:
        print(f'\n⚠️  {target_col}: слишком мало валидных образцов ({len(sub)}), пропуск')
        continue

    X      = sub[FEATURE_COLS].values
    y_raw  = sub[target_col].values
    y_log2 = np.log2(y_raw)
    mol_idx = sub['mol_idx'].values

    idx_arr = np.arange(len(sub))
    train_idx, test_idx = train_test_split(
        idx_arr, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_log2[train_idx], y_log2[test_idx]
    mol_test = mol_idx[test_idx]

    model = CatBoostRegressor(**CB_KW)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    pr, _ = pearsonr(y_test, y_pred)

    print(f'\n{target_col}')
    print(f'  N_train={len(y_train)}  N_test={len(y_test)}')
    print(f'  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  Pearson={pr:.4f}')

    per_target_df = pd.DataFrame({
        'mol_idx':       mol_test,
        'organism':      organism,
        'metric_type':   short_name,
        'target_column': target_col,
        'y_true_log2':   y_test,
        'y_pred_log2':   y_pred,
        'y_true_raw':    2.0 ** y_test,
        'y_pred_raw':    2.0 ** y_pred,
    })
    out_path = DESC_OUT_DIR / f'test_pred__{tag}.csv'
    per_target_df.to_csv(out_path, index=False)
    print(f'  ✓ {out_path.name}')

    desc_all_records.append(per_target_df)
    desc_metrics_rows.append({
        'Approach': 'descriptors',
        'Organism': organism, 'Target': short_name,
        'N_train': len(y_train), 'N_test': len(y_test),
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'Pearson_r': pr,
    })

desc_all_df = pd.concat(desc_all_records, ignore_index=True)
desc_all_df.to_csv(DESC_OUT_DIR / 'test_pred__ALL_targets_concat.csv', index=False)

desc_metrics_df = pd.DataFrame(desc_metrics_rows)
desc_metrics_df.to_csv(DESC_OUT_DIR / 'metrics_per_target.csv', index=False)

y_true_all = desc_all_df['y_true_log2'].values
y_pred_all = desc_all_df['y_pred_log2'].values
desc_overall = {
    'Approach': 'descriptors',
    'N_test_total': len(y_true_all),
    'MAE':  mean_absolute_error(y_true_all, y_pred_all),
    'RMSE': np.sqrt(mean_squared_error(y_true_all, y_pred_all)),
    'R2':   r2_score(y_true_all, y_pred_all),
    'Pearson_r': pearsonr(y_true_all, y_pred_all)[0],
}
pd.DataFrame([desc_overall]).to_csv(DESC_OUT_DIR / 'metrics_overall.csv', index=False)

print('\n=== DESCRIPTORS — overall (concat) ===')
for k, v in desc_overall.items():
    print(f'  {k}: {v}')
print(f'\n✓ {DESC_OUT_DIR}')

In [ ]:
# %%
emb_all_records  = []
emb_metrics_rows = []

for target_col in TARGET_COLUMNS:
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    tag = f"{organism.replace(' ', '_').replace('.', '')}_{short_name}"

    valid_mask = ~target[target_col].isna()
    valid_idx_arr = np.where(valid_mask.values)[0]

    if len(valid_idx_arr) < 10:
        print(f'\n⚠️  {target_col}: слишком мало валидных образцов ({len(valid_idx_arr)}), пропуск')
        continue

    X      = molecular_embeddings[valid_idx_arr]
    y_raw  = target[target_col].values[valid_idx_arr]
    y_log2 = np.log2(y_raw)

    pos_arr = np.arange(len(valid_idx_arr))
    train_pos, test_pos = train_test_split(
        pos_arr, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    X_train, X_test = X[train_pos], X[test_pos]
    y_train, y_test = y_log2[train_pos], y_log2[test_pos]
    mol_test = valid_idx_arr[test_pos]

    model = CatBoostRegressor(**CB_KW)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    pr, _ = pearsonr(y_test, y_pred)

    print(f'\n{target_col}')
    print(f'  N_train={len(y_train)}  N_test={len(y_test)}')
    print(f'  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  Pearson={pr:.4f}')

    per_target_df = pd.DataFrame({
        'mol_idx':       mol_test,
        'organism':      organism,
        'metric_type':   short_name,
        'target_column': target_col,
        'y_true_log2':   y_test,
        'y_pred_log2':   y_pred,
        'y_true_raw':    2.0 ** y_test,
        'y_pred_raw':    2.0 ** y_pred,
    })
    out_path = EMB_OUT_DIR / f'test_pred__{tag}.csv'
    per_target_df.to_csv(out_path, index=False)
    print(f'  ✓ {out_path.name}')

    emb_all_records.append(per_target_df)
    emb_metrics_rows.append({
        'Approach': 'embeddings',
        'Organism': organism, 'Target': short_name,
        'N_train': len(y_train), 'N_test': len(y_test),
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'Pearson_r': pr,
    })

emb_all_df = pd.concat(emb_all_records, ignore_index=True)
emb_all_df.to_csv(EMB_OUT_DIR / 'test_pred__ALL_targets_concat.csv', index=False)

emb_metrics_df = pd.DataFrame(emb_metrics_rows)
emb_metrics_df.to_csv(EMB_OUT_DIR / 'metrics_per_target.csv', index=False)

y_true_all = emb_all_df['y_true_log2'].values
y_pred_all = emb_all_df['y_pred_log2'].values
emb_overall = {
    'Approach': 'embeddings',
    'N_test_total': len(y_true_all),
    'MAE':  mean_absolute_error(y_true_all, y_pred_all),
    'RMSE': np.sqrt(mean_squared_error(y_true_all, y_pred_all)),
    'R2':   r2_score(y_true_all, y_pred_all),
    'Pearson_r': pearsonr(y_true_all, y_pred_all)[0],
}
pd.DataFrame([emb_overall]).to_csv(EMB_OUT_DIR / 'metrics_overall.csv', index=False)

print('\n=== EMBEDDINGS — overall (concat) ===')
for k, v in emb_overall.items():
    print(f'  {k}: {v}')
print(f'\n✓ {EMB_OUT_DIR}')

In [ ]:
# %%
summary = pd.concat([desc_metrics_df, emb_metrics_df], ignore_index=True)
print('Per-target:')
print(summary.to_string(index=False))

overall = pd.DataFrame([desc_overall, emb_overall])
print('\nOverall (concat across all 4 targets):')
print(overall.to_string(index=False))

summary.to_csv(BASE / 'comparison_per_target.csv', index=False)
overall.to_csv(BASE / 'comparison_overall.csv', index=False)
print(f'\n✓ {BASE}/comparison_per_target.csv')
print(f'✓ {BASE}/comparison_overall.csv')

# Multitask on the new эмбеддингм

In [ ]:
import os
# %%
from openai import OpenAI

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
EMBEDDING_MODEL = "openai/text-embedding-3-small"

client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

def get_embedding(text, model=EMBEDDING_MODEL):
    text = text.replace("\n", " ")
    resp = client.embeddings.create(input=[text], model=model)
    return resp.data[0].embedding

CLASS_DESCRIPTIONS = {
    'S. aureus ATCC 43300_MIC': (
        "Minimum Inhibitory Concentration against Staphylococcus aureus ATCC 43300, "
        "a methicillin-resistant Gram-positive bacterium. MIC measures the lowest "
        "concentration that inhibits visible bacterial growth."
    ),
    'S. aureus ATCC 43300_MBC': (
        "Minimum Bactericidal Concentration against Staphylococcus aureus ATCC 43300, "
        "a methicillin-resistant Gram-positive bacterium. MBC measures the lowest "
        "concentration that kills 99.9% of bacteria."
    ),
    'E. coli ATCC 25922_MIC': (
        "Minimum Inhibitory Concentration against Escherichia coli ATCC 25922, "
        "a Gram-negative reference strain. MIC measures the lowest concentration "
        "that inhibits visible bacterial growth."
    ),
    'E. coli ATCC 25922_MBC': (
        "Minimum Bactericidal Concentration against Escherichia coli ATCC 25922, "
        "a Gram-negative reference strain. MBC measures the lowest concentration "
        "that kills 99.9% of bacteria."
    ),
}

# проверка соединения + получение эмбеддингов
print('Получение OpenAI эмбеддингов меток через OpenRouter...')
class_embeddings = {}
for key, desc in CLASS_DESCRIPTIONS.items():
    emb = get_embedding(desc)
    class_embeddings[key] = np.array(emb, dtype=np.float32)
    print(f'  {key}: dim={len(emb)}')

openai_emb_dim = len(next(iter(class_embeddings.values())))
print(f'\n✓ OpenAI label embeddings: {openai_emb_dim}-dim')

In [ ]:
# %%
from catboost import Pool

# Папки вывода
DESC_CAT_OUT_DIR = BASE / 'descriptors_cat_test_predictions'
DESC_OAI_OUT_DIR = BASE / 'descriptors_openai_test_predictions'
EMB_CAT_OUT_DIR  = BASE / 'embeddings_cat_test_predictions'
EMB_OAI_OUT_DIR  = BASE / 'embeddings_openai_test_predictions'
for d in [DESC_CAT_OUT_DIR, DESC_OAI_OUT_DIR, EMB_CAT_OUT_DIR, EMB_OAI_OUT_DIR]:
    d.mkdir(exist_ok=True)

# --- Multi-task на ЭМБЕДДИНГАХ ---
emb_dim = molecular_embeddings.shape[1]
print(f'molecular_embeddings dim = {emb_dim}')

emb_records = []
for target_col in TARGET_COLUMNS:
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    key = f'{organism}_{short_name}'
    valid_idx = np.where(~target[target_col].isna().values)[0]
    for mi in valid_idx:
        emb_records.append({
            'mol_idx': int(mi),
            'organism': organism,
            'metric_type': short_name,
            'target_column': target_col,
            'class_key': key,
            'y_log2': float(np.log2(target[target_col].values[mi])),
        })
emb_pairs_df = pd.DataFrame(emb_records).reset_index(drop=True)
print(f'Emb multi-task: {len(emb_pairs_df)} пар')

# --- Multi-task на ДЕСКРИПТОРАХ ---
# берём только валидные строки (где есть фичи и есть таргет)
desc_records_rows = []
for target_col in TARGET_COLUMNS:
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    key = f'{organism}_{short_name}'

    cols_needed = FEATURE_COLS + [target_col]
    valid_mask = desc_df[cols_needed].notna().all(axis=1)
    sub = desc_df.loc[valid_mask, cols_needed].copy()
    sub['mol_idx']       = sub.index.values
    sub['organism']      = organism
    sub['metric_type']   = short_name
    sub['target_column'] = target_col
    sub['class_key']     = key
    sub['y_log2']        = np.log2(sub[target_col].values)
    desc_records_rows.append(sub)

desc_pairs_df = pd.concat(desc_records_rows, ignore_index=True)
print(f'Desc multi-task: {len(desc_pairs_df)} пар')


def make_split(n_pairs):
    return train_test_split(np.arange(n_pairs), test_size=TEST_SIZE, random_state=RANDOM_STATE)


def per_target_metrics(test_df, approach_name):
    rows = []
    for tc in TARGET_COLUMNS:
        sub = test_df[test_df['target_column'] == tc]
        if len(sub) < 2:
            continue
        short_name = tc.split(' Activity ')[1].replace(', мг/л', '')
        organism   = tc.split(' Activity ')[0]
        yt, yp = sub['y_true_log2'].values, sub['y_pred_log2'].values
        rows.append({
            'Approach': approach_name,
            'Organism': organism, 'Target': short_name,
            'N_test': len(sub),
            'MAE':  mean_absolute_error(yt, yp),
            'RMSE': np.sqrt(mean_squared_error(yt, yp)),
            'R2':   r2_score(yt, yp),
            'Pearson_r': pearsonr(yt, yp)[0],
        })
    return pd.DataFrame(rows)


def save_multitask_outputs(test_df, approach_name, out_dir):
    cols = ['mol_idx','organism','metric_type','target_column',
            'y_true_log2','y_pred_log2','y_true_raw','y_pred_raw']
    for tc in TARGET_COLUMNS:
        sub = test_df[test_df['target_column'] == tc]
        if len(sub) < 2:
            continue
        short_name = tc.split(' Activity ')[1].replace(', мг/л', '')
        organism   = tc.split(' Activity ')[0]
        tag = f"{organism.replace(' ', '_').replace('.', '')}_{short_name}"
        sub[cols].to_csv(out_dir / f'test_pred__{tag}.csv', index=False)

    test_df[cols].to_csv(out_dir / 'test_pred__ALL_targets_concat.csv', index=False)
    pt = per_target_metrics(test_df, approach_name)
    pt.to_csv(out_dir / 'metrics_per_target.csv', index=False)

    yt, yp = test_df['y_true_log2'].values, test_df['y_pred_log2'].values
    overall = {
        'Approach': approach_name,
        'N_test_total': len(test_df),
        'MAE':  mean_absolute_error(yt, yp),
        'RMSE': np.sqrt(mean_squared_error(yt, yp)),
        'R2':   r2_score(yt, yp),
        'Pearson_r': pearsonr(yt, yp)[0],
    }
    pd.DataFrame([overall]).to_csv(out_dir / 'metrics_overall.csv', index=False)
    return pt, overall

In [ ]:
# %%
# Сплит для дескрипторного multi-task
train_idx_d, test_idx_d = make_split(len(desc_pairs_df))
y_full_d = desc_pairs_df['y_log2'].values

# === 3) desc + categorical ===
print('\n--- desc + categorical ---')
X_d_cat = desc_pairs_df[FEATURE_COLS].copy()
X_d_cat['organism']    = desc_pairs_df['organism'].values
X_d_cat['metric_type'] = desc_pairs_df['metric_type'].values

X_tr = X_d_cat.iloc[train_idx_d].reset_index(drop=True)
X_te = X_d_cat.iloc[test_idx_d].reset_index(drop=True)
y_tr = y_full_d[train_idx_d]
y_te = y_full_d[test_idx_d]

train_pool = Pool(X_tr, label=y_tr, cat_features=['organism', 'metric_type'])
test_pool  = Pool(X_te, label=y_te, cat_features=['organism', 'metric_type'])
m = CatBoostRegressor(**CB_KW); m.fit(train_pool)
y_pred = m.predict(test_pool)

meta = desc_pairs_df.iloc[test_idx_d].reset_index(drop=True).copy()
meta['y_true_log2'] = y_te
meta['y_pred_log2'] = y_pred
meta['y_true_raw']  = 2.0 ** y_te
meta['y_pred_raw']  = 2.0 ** y_pred
desc_cat_metrics_df, desc_cat_overall = save_multitask_outputs(meta, 'desc+categorical', DESC_CAT_OUT_DIR)
print(f"Overall  MAE={desc_cat_overall['MAE']:.4f}  R²={desc_cat_overall['R2']:.4f}  "
      f"Pearson={desc_cat_overall['Pearson_r']:.4f}")

# === 4) desc + OpenAI ===
print('\n--- desc + openai ---')
label_emb_d = np.stack(
    [class_embeddings[k] for k in desc_pairs_df['class_key'].values]
).astype(np.float32)
X_d_oai = np.hstack([desc_pairs_df[FEATURE_COLS].values, label_emb_d])

X_tr = X_d_oai[train_idx_d]; X_te = X_d_oai[test_idx_d]
y_tr = y_full_d[train_idx_d]; y_te = y_full_d[test_idx_d]

m = CatBoostRegressor(**CB_KW); m.fit(X_tr, y_tr)
y_pred = m.predict(X_te)

meta = desc_pairs_df.iloc[test_idx_d].reset_index(drop=True).copy()
meta['y_true_log2'] = y_te
meta['y_pred_log2'] = y_pred
meta['y_true_raw']  = 2.0 ** y_te
meta['y_pred_raw']  = 2.0 ** y_pred
desc_oai_metrics_df, desc_oai_overall = save_multitask_outputs(meta, 'desc+openai', DESC_OAI_OUT_DIR)
print(f"Overall  MAE={desc_oai_overall['MAE']:.4f}  R²={desc_oai_overall['R2']:.4f}  "
      f"Pearson={desc_oai_overall['Pearson_r']:.4f}")

In [ ]:
# %%
train_idx_e, test_idx_e = make_split(len(emb_pairs_df))
y_full_e = emb_pairs_df['y_log2'].values
mol_emb_per_pair = molecular_embeddings[emb_pairs_df['mol_idx'].values]
emb_cols = [f'emb_{i}' for i in range(emb_dim)]

# === 5) emb + categorical ===
print('\n--- emb + categorical ---')
X_e_cat = pd.DataFrame(mol_emb_per_pair, columns=emb_cols)
X_e_cat['organism']    = emb_pairs_df['organism'].values
X_e_cat['metric_type'] = emb_pairs_df['metric_type'].values

X_tr = X_e_cat.iloc[train_idx_e].reset_index(drop=True)
X_te = X_e_cat.iloc[test_idx_e].reset_index(drop=True)
y_tr = y_full_e[train_idx_e]; y_te = y_full_e[test_idx_e]

train_pool = Pool(X_tr, label=y_tr, cat_features=['organism', 'metric_type'])
test_pool  = Pool(X_te, label=y_te, cat_features=['organism', 'metric_type'])
m = CatBoostRegressor(**CB_KW); m.fit(train_pool)
y_pred = m.predict(test_pool)

meta = emb_pairs_df.iloc[test_idx_e].reset_index(drop=True).copy()
meta['y_true_log2'] = y_te
meta['y_pred_log2'] = y_pred
meta['y_true_raw']  = 2.0 ** y_te
meta['y_pred_raw']  = 2.0 ** y_pred
emb_cat_metrics_df, emb_cat_overall = save_multitask_outputs(meta, 'emb+categorical', EMB_CAT_OUT_DIR)
print(f"Overall  MAE={emb_cat_overall['MAE']:.4f}  R²={emb_cat_overall['R2']:.4f}  "
      f"Pearson={emb_cat_overall['Pearson_r']:.4f}")

# === 6) emb + OpenAI ===
print('\n--- emb + openai ---')
label_emb_e = np.stack(
    [class_embeddings[k] for k in emb_pairs_df['class_key'].values]
).astype(np.float32)
X_e_oai = np.hstack([mol_emb_per_pair, label_emb_e])

X_tr = X_e_oai[train_idx_e]; X_te = X_e_oai[test_idx_e]
y_tr = y_full_e[train_idx_e]; y_te = y_full_e[test_idx_e]

m = CatBoostRegressor(**CB_KW); m.fit(X_tr, y_tr)
y_pred = m.predict(X_te)

meta = emb_pairs_df.iloc[test_idx_e].reset_index(drop=True).copy()
meta['y_true_log2'] = y_te
meta['y_pred_log2'] = y_pred
meta['y_true_raw']  = 2.0 ** y_te
meta['y_pred_raw']  = 2.0 ** y_pred
emb_oai_metrics_df, emb_oai_overall = save_multitask_outputs(meta, 'emb+openai', EMB_OAI_OUT_DIR)
print(f"Overall  MAE={emb_oai_overall['MAE']:.4f}  R²={emb_oai_overall['R2']:.4f}  "
      f"Pearson={emb_oai_overall['Pearson_r']:.4f}")

In [ ]:
# %%
from catboost import Pool

MULTI_CAT_OUT_DIR = BASE / 'embeddings_cat_test_predictions'
MULTI_CAT_OUT_DIR.mkdir(exist_ok=True)

emb_dim = molecular_embeddings.shape[1]
print(f'molecular_embeddings dim = {emb_dim}')

# --- Собираем multi-task датасет ---
records = []
for target_col in TARGET_COLUMNS:
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    key = f'{organism}_{short_name}'

    valid_idx = np.where(~target[target_col].isna().values)[0]
    for mi in valid_idx:
        records.append({
            'mol_idx':       int(mi),
            'organism':      organism,
            'metric_type':   short_name,
            'target_column': target_col,
            'class_key':     key,
            'y_log2':        float(np.log2(target[target_col].values[mi])),
        })

pairs_df = pd.DataFrame(records).reset_index(drop=True)
print(f'Multi-task датасет: {len(pairs_df)} пар (mol, target)')

# --- Фичи: molecular_embeddings[mol_idx] + organism + metric_type ---
mol_emb_per_pair = molecular_embeddings[pairs_df['mol_idx'].values]
emb_cols = [f'emb_{i}' for i in range(emb_dim)]
X_full = pd.DataFrame(mol_emb_per_pair, columns=emb_cols)
X_full['organism']    = pairs_df['organism'].values
X_full['metric_type'] = pairs_df['metric_type'].values
y_full = pairs_df['y_log2'].values

# --- Split (как в прошлых экспериментах) ---
idx_arr = np.arange(len(pairs_df))
train_idx, test_idx = train_test_split(
    idx_arr, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

X_train = X_full.iloc[train_idx].reset_index(drop=True)
X_test  = X_full.iloc[test_idx].reset_index(drop=True)
y_train = y_full[train_idx]
y_test  = y_full[test_idx]
meta_test = pairs_df.iloc[test_idx].reset_index(drop=True)

print(f'N_train={len(train_idx)}  N_test={len(test_idx)}')

train_pool = Pool(data=X_train, label=y_train, cat_features=['organism', 'metric_type'])
test_pool  = Pool(data=X_test,  label=y_test,  cat_features=['organism', 'metric_type'])

model_cat = CatBoostRegressor(**CB_KW)
model_cat.fit(train_pool)
y_pred_cat = model_cat.predict(test_pool)

mae  = mean_absolute_error(y_test, y_pred_cat)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_cat))
r2   = r2_score(y_test, y_pred_cat)
pr, _ = pearsonr(y_test, y_pred_cat)
print(f'\nOverall: MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  Pearson={pr:.4f}')

# --- Сохранение per-target и concat ---
test_df = meta_test.copy()
test_df['y_true_log2'] = y_test
test_df['y_pred_log2'] = y_pred_cat
test_df['y_true_raw']  = 2.0 ** y_test
test_df['y_pred_raw']  = 2.0 ** y_pred_cat

cat_metrics_rows = []
for target_col in TARGET_COLUMNS:
    sub = test_df[test_df['target_column'] == target_col]
    if len(sub) < 2:
        continue
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    tag = f"{organism.replace(' ', '_').replace('.', '')}_{short_name}"

    out_path = MULTI_CAT_OUT_DIR / f'test_pred__{tag}.csv'
    sub[['mol_idx','organism','metric_type','target_column',
         'y_true_log2','y_pred_log2','y_true_raw','y_pred_raw']].to_csv(out_path, index=False)

    yt, yp = sub['y_true_log2'].values, sub['y_pred_log2'].values
    cat_metrics_rows.append({
        'Approach': 'emb+categorical',
        'Organism': organism, 'Target': short_name,
        'N_test': len(sub),
        'MAE':  mean_absolute_error(yt, yp),
        'RMSE': np.sqrt(mean_squared_error(yt, yp)),
        'R2':   r2_score(yt, yp),
        'Pearson_r': pearsonr(yt, yp)[0],
    })
    print(f'  ✓ {out_path.name}  (N={len(sub)})')

test_df[['mol_idx','organism','metric_type','target_column',
         'y_true_log2','y_pred_log2','y_true_raw','y_pred_raw']].to_csv(
    MULTI_CAT_OUT_DIR / 'test_pred__ALL_targets_concat.csv', index=False
)

cat_metrics_df = pd.DataFrame(cat_metrics_rows)
cat_metrics_df.to_csv(MULTI_CAT_OUT_DIR / 'metrics_per_target.csv', index=False)

cat_overall = {
    'Approach': 'emb+categorical',
    'N_test_total': len(test_df),
    'MAE':  mae, 'RMSE': rmse, 'R2': r2, 'Pearson_r': pr,
}
pd.DataFrame([cat_overall]).to_csv(MULTI_CAT_OUT_DIR / 'metrics_overall.csv', index=False)
print(f'\n✓ {MULTI_CAT_OUT_DIR}')

In [ ]:
# %%
MULTI_OAI_OUT_DIR = BASE / 'embeddings_openai_test_predictions'
MULTI_OAI_OUT_DIR.mkdir(exist_ok=True)

# pairs_df, mol_emb_per_pair, y_full из предыдущей ячейки переиспользуем
label_emb_per_pair = np.stack(
    [class_embeddings[k] for k in pairs_df['class_key'].values]
).astype(np.float32)

X_full_oai = np.hstack([mol_emb_per_pair, label_emb_per_pair])
print(f'X_full_oai: {X_full_oai.shape}  ({emb_dim} mol + {openai_emb_dim} openai)')

# Тот же split (тот же seed, тот же размер → те же индексы)
train_idx_oai, test_idx_oai = train_test_split(
    np.arange(len(pairs_df)), test_size=TEST_SIZE, random_state=RANDOM_STATE
)
# sanity-check: train_idx_oai должен совпадать с train_idx из 5.6
assert np.array_equal(train_idx_oai, train_idx), 'split разъехался — проверь seed'

X_train_oai = X_full_oai[train_idx_oai]
X_test_oai  = X_full_oai[test_idx_oai]
y_train_oai = y_full[train_idx_oai]
y_test_oai  = y_full[test_idx_oai]
meta_test_oai = pairs_df.iloc[test_idx_oai].reset_index(drop=True)

print(f'N_train={len(train_idx_oai)}  N_test={len(test_idx_oai)}')

model_oai = CatBoostRegressor(**CB_KW)
model_oai.fit(X_train_oai, y_train_oai)
y_pred_oai = model_oai.predict(X_test_oai)

mae  = mean_absolute_error(y_test_oai, y_pred_oai)
rmse = np.sqrt(mean_squared_error(y_test_oai, y_pred_oai))
r2   = r2_score(y_test_oai, y_pred_oai)
pr, _ = pearsonr(y_test_oai, y_pred_oai)
print(f'\nOverall: MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  Pearson={pr:.4f}')

# --- Сохранение per-target и concat ---
test_df = meta_test_oai.copy()
test_df['y_true_log2'] = y_test_oai
test_df['y_pred_log2'] = y_pred_oai
test_df['y_true_raw']  = 2.0 ** y_test_oai
test_df['y_pred_raw']  = 2.0 ** y_pred_oai

oai_metrics_rows = []
for target_col in TARGET_COLUMNS:
    sub = test_df[test_df['target_column'] == target_col]
    if len(sub) < 2:
        continue
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    tag = f"{organism.replace(' ', '_').replace('.', '')}_{short_name}"

    out_path = MULTI_OAI_OUT_DIR / f'test_pred__{tag}.csv'
    sub[['mol_idx','organism','metric_type','target_column',
         'y_true_log2','y_pred_log2','y_true_raw','y_pred_raw']].to_csv(out_path, index=False)

    yt, yp = sub['y_true_log2'].values, sub['y_pred_log2'].values
    oai_metrics_rows.append({
        'Approach': 'emb+openai',
        'Organism': organism, 'Target': short_name,
        'N_test': len(sub),
        'MAE':  mean_absolute_error(yt, yp),
        'RMSE': np.sqrt(mean_squared_error(yt, yp)),
        'R2':   r2_score(yt, yp),
        'Pearson_r': pearsonr(yt, yp)[0],
    })
    print(f'  ✓ {out_path.name}  (N={len(sub)})')

test_df[['mol_idx','organism','metric_type','target_column',
         'y_true_log2','y_pred_log2','y_true_raw','y_pred_raw']].to_csv(
    MULTI_OAI_OUT_DIR / 'test_pred__ALL_targets_concat.csv', index=False
)

oai_metrics_df = pd.DataFrame(oai_metrics_rows)
oai_metrics_df.to_csv(MULTI_OAI_OUT_DIR / 'metrics_per_target.csv', index=False)

oai_overall = {
    'Approach': 'emb+openai',
    'N_test_total': len(test_df),
    'MAE':  mae, 'RMSE': rmse, 'R2': r2, 'Pearson_r': pr,
}
pd.DataFrame([oai_overall]).to_csv(MULTI_OAI_OUT_DIR / 'metrics_overall.csv', index=False)
print(f'\n✓ {MULTI_OAI_OUT_DIR}')

In [ ]:
# %%
# Все метрики per-target в один DataFrame
all_per_target = pd.concat([
    desc_metrics_df, emb_metrics_df,
    desc_cat_metrics_df, desc_oai_metrics_df,
    emb_cat_metrics_df,  emb_oai_metrics_df,
], ignore_index=True)

# Короткие коды классов: SA_MIC, SA_MBC, EC_MIC, EC_MBC
def short_class(row):
    org = 'SA' if row['Organism'].startswith('S.') else 'EC'
    return f"{org}_{row['Target']}"

all_per_target['Class'] = all_per_target.apply(short_class, axis=1)
CLASS_ORDER = ['SA_MIC', 'SA_MBC', 'EC_MIC', 'EC_MBC']
APPROACH_ORDER = ['descriptors', 'embeddings',
                  'desc+categorical', 'desc+openai',
                  'emb+categorical', 'emb+openai']

# --- Сводные таблицы по каждой метрике: строки=подходы, столбцы=4 класса ---
def pivot_metric(metric):
    pv = all_per_target.pivot_table(
        index='Approach', columns='Class', values=metric, aggfunc='first'
    )
    pv = pv.reindex(index=APPROACH_ORDER, columns=CLASS_ORDER)
    return pv

print('\n' + '='*70)
print('РАСКЛАД ПО КЛАССАМ:  SA_MIC | SA_MBC | EC_MIC | EC_MBC')
print('='*70)

for metric in ['MAE', 'RMSE', 'R2', 'Pearson_r']:
    pv = pivot_metric(metric)
    print(f'\n--- {metric} ---')
    print(pv.round(4).to_string())
    pv.to_csv(BASE / f'comparison_by_class__{metric}.csv')

# --- Общий overall (concat across all 4 targets) ---
overall = pd.DataFrame([
    desc_overall, emb_overall,
    desc_cat_overall, desc_oai_overall,
    emb_cat_overall,  emb_oai_overall,
])

print('\n' + '='*70)
print('OVERALL (concat across all 4 targets)')
print('='*70)
print(overall[['Approach','N_test_total','MAE','RMSE','R2','Pearson_r']].round(4).to_string(index=False))

# --- Сохранение ---
all_per_target.to_csv(BASE / 'comparison_per_target_LONG.csv', index=False)
overall.to_csv(BASE / 'comparison_overall.csv', index=False)
print(f'\n✓ {BASE}/comparison_per_target_LONG.csv')
print(f'✓ {BASE}/comparison_overall.csv')
print(f'✓ {BASE}/comparison_by_class__<MAE|RMSE|R2|Pearson_r>.csv')

In [ ]:
# %%
summary = pd.concat([
    desc_metrics_df, emb_metrics_df, cat_metrics_df, oai_metrics_df
], ignore_index=True)
print('Per-target:')
print(summary.to_string(index=False))

overall = pd.DataFrame([desc_overall, emb_overall, cat_overall, oai_overall])
print('\nOverall (concat across all 4 targets):')
print(overall.to_string(index=False))

summary.to_csv(BASE / 'comparison_per_target.csv', index=False)
overall.to_csv(BASE / 'comparison_overall.csv', index=False)
print(f'\n✓ {BASE}/comparison_per_target.csv')
print(f'✓ {BASE}/comparison_overall.csv')